# Lab 1: Qubits, gates, and interference

**Audience:** comfortable with Python and basic linear algebra; no prior quantum computing required for Lab 1.

## Learning objectives

- Represent a qubit with normalized complex amplitudes and calculate measurement probabilities.
- Build and read circuits containing X, H, Z, and rotation gates.
- Distinguish relative phase, interference, exact probabilities, and sampled frequencies.
- Prepare a qubit with a chosen measurement probability.

## How to use this notebook

Open [Google Colab](https://colab.research.google.com/), choose **File → Upload notebook**, and upload this file. Connect to a standard Python CPU runtime. Run cells in order with Shift+Enter. No IBM account, token, GPU, or paid service is needed. Internet access is needed for package installation only. If changing an already-used package version, restart the session after installation and rerun from the imports cell. These notebooks target Python 3.10–3.13.

For each experiment: **explain → predict → run → interpret → modify**. Write predictions and observations in a new text cell or your notes. Exercise starter cells are working examples, not finished answers to every requested change. Hints and solutions are collapsed below the prompts. Keep optional extensions until after the core lab.

## Agenda

| Activity |
|---|
| Setup and circuit model |
| States, amplitudes, measurement; Exercise 1 |
| Gates and rotations; Exercise 2 |
| Break and checkpoint |
| Phase and interference; Exercise 3 |
| Guided state preparation; Exercise 4 |
| Exit questions |

## Prerequisite recap

A Python list can store vector components; NumPy computes norms and matrix products. A complex number has a magnitude and a phase. We will introduce the quantum meaning of these quantities here. No physics formulas need to be memorized.

**Working together:** compare predictions with a partner before running code. If you finish early, try an optional extension.


## Setup

The first cell installs a fixed teaching environment. Installation may take a few minutes. The helper functions below build explicit classical measurement registers and use 2,048 shots with seed 42. A shot means one fresh preparation and measurement of the circuit. Fixed seeds make classroom results repeatable; they do not remove sampling uncertainty.

`exact` inspects a simulated state without measurement. `sample` returns finite-shot counts. `compare` plots both. Access to all amplitudes is a simulator convenience: a physical quantum computer does not hand you its state vector.


In [ ]:
%pip install -q qiskit==2.2.3 numpy==2.2.6 matplotlib==3.10.3 pylatexenc==2.10 sympy==1.14.0


In [ ]:
import sys
from importlib.metadata import version
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, DensityMatrix
from qiskit.primitives import StatevectorSampler

SEED, SHOTS = 42, 2048
print('Python:', sys.version.split()[0])
for package in ['qiskit', 'numpy', 'matplotlib', 'pylatexenc', 'sympy']:
    print(f'{package}: {version(package)}')

def exact(circuit):
    # Supply a circuit WITHOUT measurements. This is simulator-only inspection.
    return Statevector.from_instruction(circuit).probabilities_dict()

def sample(circuit, qubits=None):
    # Copy so that the original circuit stays available for state inspection.
    qubits = list(range(circuit.num_qubits)) if qubits is None else list(qubits)
    measured = circuit.copy()
    readout = ClassicalRegister(len(qubits), 'readout')
    measured.add_register(readout)
    measured.measure(qubits, readout)
    sampler = StatevectorSampler(seed=SEED)
    return sampler.run([measured], shots=SHOTS).result()[0].data.readout.get_counts()

def compare(circuit, title):
    probabilities, counts = exact(circuit), sample(circuit)
    labels = [format(i, f'0{circuit.num_qubits}b') for i in range(2**circuit.num_qubits)]
    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.bar(x - .18, [probabilities.get(k, 0) for k in labels], .36, label='Exact probability')
    ax.bar(x + .18, [counts.get(k, 0) / SHOTS for k in labels], .36, label=f'Sampled frequency ({SHOTS} shots)')
    ax.set(xticks=x, xticklabels=labels, ylim=(0, 1.12), ylabel='Probability / frequency', xlabel='Measured bit string', title=title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
    print('Counts:', counts)
    return counts

def show(circuit):
    display(circuit.draw('mpl', fold=70))
    plt.show()


## From bits to circuits

A classical bit takes value 0 or 1. A qubit has basis states `|0⟩=(1,0)ᵀ` and `|1⟩=(0,1)ᵀ`. The symbols | and ⟩ form **ket notation**, which names a state vector. A general pure qubit state is `|ψ⟩=α|0⟩+β|1⟩`.

In these simulations, a circuit starts with every qubit in `|0⟩`. Gates change the state; measurement produces classical bits. Read a Qiskit circuit left to right; wires identify qubits, boxes identify gates, and measurement arrows point to classical storage. A quantum circuit is not a program that reads out all possible answers at once.

**Predict:** What should a circuit with no gates return? Run this first circuit and identify the classical register in its diagram.


In [ ]:
first = QuantumCircuit(QuantumRegister(1, 'q'), ClassicalRegister(1, 'result'))
first.measure(0, 0)
show(first)
result = StatevectorSampler(seed=SEED).run([first], shots=SHOTS).result()[0]
print(result.data.result.get_counts())


## Amplitudes become probabilities

Normalization requires `|α|²+|β|²=1`. In a computational-basis (Z-basis) measurement, `P(0)=|α|²` and `P(1)=|β|²`: the **Born rule**. Amplitudes can be negative or complex; probabilities cannot. After an ideal measurement with outcome 0, the state is `|0⟩` (and similarly for 1). To estimate a distribution, repeat the entire preparation, not measurements of an unchanged copy.

**Predict:** Do amplitudes `√0.3` and `i√0.7` give probabilities 0.3 and 0.7? Does the imaginary unit change the second probability?


In [ ]:
amplitudes = np.array([np.sqrt(.3), 1j * np.sqrt(.7)])
print('Amplitudes:', amplitudes)
print('Norm squared:', np.vdot(amplitudes, amplitudes).real)
print('Probabilities:', np.abs(amplitudes)**2)
assert np.isclose(np.linalg.norm(amplitudes), 1)
state = Statevector(amplitudes)
print('State vector:', state.data)


### Exercise 1 — Amplitudes are not probabilities

Change the baseline state to amplitudes `1/2` and `√3/2`. Predict both probabilities and check normalization. Then multiply the entire vector by `i`: what changes in its measurement probabilities?

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

Use `np.abs(vector)**2`. A common phase multiplying both amplitudes is a global phase.

</details>

<details><summary>Worked solution — open after attempting</summary>

Use `vector = np.array([.5, np.sqrt(3)/2])`. The probabilities are 0.25 and 0.75 and the squared norm is 1. `np.abs(1j * vector)**2` is identical. Global phase has no observable effect.

</details>


In [ ]:
vector = np.array([1, 1j]) / np.sqrt(2)  # Working baseline: equal probabilities
print('Norm:', np.linalg.norm(vector))
print('Probabilities:', np.abs(vector)**2)
print('After global phase:', np.abs(1j * vector)**2)


## Gates transform amplitudes

Gates are unitary matrices: they preserve normalization and are reversible before measurement.

```text
X = [0  1]    H = (1/√2) [1   1]    Z = [1   0]
    [1  0]               [1  −1]        [0  −1]
```

X swaps the basis amplitudes. H sends `|0⟩` to `|+⟩=(|0⟩+|1⟩)/√2` and `|1⟩` to `|-⟩=(|0⟩-|1⟩)/√2`. Z flips the sign of the `|1⟩` amplitude. Sign matters when amplitudes later interfere.

**Predict:** Compare no gate, X, H, and H followed by Z. Which can you distinguish with Z-basis counts alone?


In [ ]:
gate_examples = {}
for name in ['Identity', 'X', 'H', 'H then Z']:
    qc = QuantumCircuit(1)
    if name == 'X': qc.x(0)
    if name.startswith('H'): qc.h(0)
    if name == 'H then Z': qc.z(0)
    gate_examples[name] = qc
    print(name, 'amplitudes:', np.round(Statevector.from_instruction(qc).data, 3))
    compare(qc, name)
show(gate_examples['H then Z'])


### Rotations and the Bloch sphere

`Rᵧ(θ)|0⟩=cos(θ/2)|0⟩+sin(θ/2)|1⟩`, so `P(1)=sin²(θ/2)`. Angles are in radians. On the Bloch sphere, poles represent basis states; equatorial states have equal Z-basis probabilities but different relative phases. The sphere represents a single qubit, not its physical trajectory.

**Predict → run → interpret:** What happens for `θ=0,π/2,π`? Modify the angle below and explain the result.


In [ ]:
theta = np.pi / 3
rotation = QuantumCircuit(1)
rotation.ry(theta, 0)
show(rotation)
display(Statevector.from_instruction(rotation).draw('bloch'))
plt.show()
compare(rotation, f'Ry({theta:.2f})')
print('Predicted P(1):', np.sin(theta/2)**2)


### Exercise 2 — Compose reversible gates

Predict the outputs of X–X and H–H starting from zero. Modify the baseline to test both. Then compare H–X and X–H using exact amplitudes: do they prepare the same state?

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

Circuit time runs left to right; the rightmost matrix acts first in a matrix product.

</details>

<details><summary>Worked solution — open after attempting</summary>

X–X and H–H both return zero. H–X prepares `|+⟩`; X–H prepares `|-⟩`. Their Z-basis probabilities are equal, but their relative phases differ. Apply one final H to distinguish them.

</details>


In [ ]:
sequence = QuantumCircuit(1)
sequence.x(0)
sequence.x(0)
show(sequence)
print('Amplitudes:', Statevector.from_instruction(sequence).data)
compare(sequence, 'Two X gates')


## Break and checkpoint

Before the break, explain to a partner: (1) why amplitude 1/2 gives probability 1/4; (2) why H followed by Z still gives roughly half zeros; (3) why 2,048 shots need not produce exactly 1,024 of each outcome. Resume at the next section after the break.


## Phase becomes visible through interference

`|+⟩` and `|-⟩` have identical Z-basis probabilities. H changes the measurement basis: H followed by Z measurement is an X-basis measurement, with bit 0 meaning `|+⟩` and bit 1 meaning `|-⟩`.

In H–H, amplitudes for outcome 1 cancel; in H–Z–H, amplitudes for outcome 0 cancel. This addition and cancellation of amplitudes is **interference**. A random classical bit with probabilities 1/2, 1/2 lacks the coherent relative phase that allows this deterministic reversal.

**Predict:** Where does the minus sign go in H–Z–H? Compare intermediate amplitudes and final counts.


In [ ]:
for use_z in [False, True]:
    qc = QuantumCircuit(1)
    qc.h(0)
    if use_z: qc.z(0)
    print('Before final H:', np.round(Statevector.from_instruction(qc).data, 3))
    qc.h(0)
    show(qc)
    compare(qc, 'H–Z–H' if use_z else 'H–H')


### Exercise 3 — A phase dial

The baseline applies H–Rz(phi)–H. Predict P(1) for phi = 0, pi/2, and pi, then run each case. Explain why Rz is invisible in Z probabilities before the final H but visible afterward.

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

Up to a global phase, Rz adds relative phase phi. The final probability is `sin²(φ/2)`.

</details>

<details><summary>Worked solution — open after attempting</summary>

The probabilities are 0, 1/2, and 1. Before the last H, the two magnitudes remain equal; after H, the relative phase determines constructive and destructive interference.

</details>


In [ ]:
phi = np.pi / 2
phase_test = QuantumCircuit(1)
phase_test.h(0)
phase_test.rz(phi, 0)
print('Before final H:', exact(phase_test))
phase_test.h(0)
compare(phase_test, 'Phase converted into probability')


## Prepare, predict, and explain

Work in pairs: one person chooses a target probability and predicts a circuit, the other checks it. Swap roles after the first successful test. Use exact probabilities to check your design and sampled counts to illustrate experimental variation.


### Exercise 4 — Prepare a biased qubit

Prepare P(1) = 0.8 using one Ry gate. Derive the angle from the formula above, then change the baseline. Explain any difference between exact probability and observed frequency. Finally, choose your own target.

Record your prediction before editing the baseline cell below. Run it, explain the result, then make the requested change.

<details><summary>Hint</summary>

Solve `p=sin²(θ/2)` with `θ=2arcsin(√p)` for `0 ≤ θ ≤ π`.

</details>

<details><summary>Worked solution — open after attempting</summary>

Set `target_p = .8` and `theta = 2*np.arcsin(np.sqrt(target_p))`, about 2.214 radians. Exact P(1) is 0.8. Sampling fluctuates; its standard deviation in frequency is `√(p(1−p)/2048)`, about 0.0088. The seed fixes one reproducible sample, not the exact probability.

</details>


In [ ]:
target_p = .25  # Change to .8 after deriving the angle
theta = 2 * np.arcsin(np.sqrt(target_p))
prepared = QuantumCircuit(1)
prepared.ry(theta, 0)
counts = compare(prepared, f'Target P(1) = {target_p}')
print('Observed P(1):', counts.get('1', 0) / SHOTS)
assert np.isclose(exact(prepared).get('1', 0), target_p)


## Exit ticket

1. What is the difference between an amplitude and a probability?
2. Why can two states have the same Z measurements yet differ physically?
3. Describe H–Z–H without saying “the qubit tries all answers.”
4. What can a simulator show that measurement on hardware cannot directly reveal?

**Optional extension:** sweep phi from 0 to 2π in Exercise 3, plot exact P(1), and overlay sin²(phi/2). Explain the extrema.

## References

- [IBM: quantum information basics](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information)
- [Qiskit QuantumCircuit](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.QuantumCircuit)
- [Qiskit StatevectorSampler](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.StatevectorSampler)
- [IBM: exact local simulation](https://quantum.cloud.ibm.com/docs/en/guides/simulate-with-qiskit-sdk-primitives)
